# 3. Qwen3-VL-2B-Instruct: SFT on Combined Dataset

This notebook trains Qwen3-VL-2B-Instruct on the combined dataset (LaTeX_OCR + 700 MathWriting-human examples) using LoRA. Zero-shot, one-shot, and single-dataset SFT results are in `3_qwen3vl_1dataset.ipynb`.

The combined training uses 2 epochs (reduced from 3) and 700 MathWriting examples (reduced from 5,000) to fit within the free Colab T4 time limit while keeping batch size and gradient accumulation identical to the single-dataset experiment for fair comparison.

Important: Qwen3-VL requires transformers >= 4.57.0. Do not run this notebook in the same Colab session as `2_smolvlm.ipynb` without restarting the runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall -y huggingface-hub transformers gradio torchao
!pip install -q huggingface-hub>=1.2.0 datasets pillow sacrebleu \
    accelerate bitsandbytes peft editdistance git+https://github.com/huggingface/transformers
!pip install -q qwen_vl_utils

Found existing installation: huggingface_hub 1.27.0
Uninstalling huggingface_hub-1.27.0:
  Successfully uninstalled huggingface_hub-1.27.0
Found existing installation: transformers 5.15.0
Uninstalling transformers-5.15.0:
  Successfully uninstalled transformers-5.15.0
Found existing installation: gradio 6.24.0
Uninstalling gradio-6.24.0:
  Successfully uninstalled gradio-6.24.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 30.9 MB/s eta 0:00:00


In [ ]:
import gc
import json
import os
import sys
import time
from collections import Counter
from glob import glob

import datasets
from datasets import Dataset, concatenate_datasets
import editdistance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from peft import LoraConfig, PeftModel, get_peft_model
from PIL import Image
from qwen_vl_utils import process_vision_info
import sacrebleu
import torch
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)

# Ensure shared modules are importable
if os.path.exists("metrics.py"):
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, "/content/drive/MyDrive/latex_ocr_project")
import data_prep
from metrics import compute_metrics, normalize_latex

In [ ]:
SEED = 42
MW_SUBSAMPLE_SIZE = 700
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
SAVE_DIR = "/content/drive/MyDrive/latex_ocr"

## 1. Data Loading

LaTeX_OCR (train/val/test) and a 700-example subsample of MathWriting-human. The combined training set has 1,864 examples. See `1_eda_and_setup.ipynb` for dataset descriptions.

In [ ]:
dataset = data_prep.load_latex_ocr()
mw_train_subsample = data_prep.load_mathwriting_subsample(
    subsample_size=MW_SUBSAMPLE_SIZE,
    random_state=SEED,
)

print("LaTeX_OCR splits:")
for split in dataset:
    print(f"  {split}: {len(dataset[split]):,} examples")
print(f"\nMathWriting subsample: {len(mw_train_subsample):,} examples")

README.md:   0%|          | 0.00/5.73k [00:00<?, ?B/s]

human_handwrite/train-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 16.2MB            

human_handwrite/train-00000-of-00001.par(…): downloading bytes:           |  0.00B            

human_handwrite/validation-00000-of-0000(…): reconstructing file:   0%|          |  0.00B /  961kB            

human_handwrite/validation-00000-of-0000(…): downloading bytes:           |  0.00B            

...

data/val-00000-of-00001-184984e66f80ed7a(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/229864 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7644 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/15674 [00:00<?, ? examples/s]

MathWriting subsample: 700 examples, columns: ['image', 'text']
LaTeX_OCR splits:
  train: 1,164 examples
  validation: 68 examples
  test: 70 examples

MathWriting subsample: 700 examples


## 2. Functions

### 2.1. Prompt Template and Inference Function

In [ ]:
def predict(images, prompt_text, model, processor, device="cpu", max_new_tokens=256,
              example_text=None):
    """
    Run inference: image(s) → LaTeX string.
    images: PIL.Image or list[PIL.Image]
    prompt_text: unused (kept for API compatibility with SmolVLM pipeline).
    example_text: if provided, a two-turn one-shot conversation is constructed
                  following Brown et al. (2020). The first image is treated as
                  the demonstration example and this text as its expected output.
    """
    INSTRUCTION = "Convert this handwritten mathematical formula to LaTeX format. Output only the LaTeX code, nothing else."

    if not isinstance(images, list):
        images = [images]

    if example_text is not None and len(images) == 2:
        # One-shot: two-turn conversation (Brown et al., 2020)
        # Turn 1: user provides example image, assistant responds with ground truth
        # Turn 2: user provides query image
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": images[0].convert("RGB")},
                {"type": "text", "text": INSTRUCTION},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": example_text}]},
            {"role": "user", "content": [
                {"type": "image", "image": images[1].convert("RGB")},
                {"type": "text", "text": INSTRUCTION},
            ]},
        ]

    else:
        # Zero-shot (or fine-tuned): single user turn with all images
        messages = [{
            "role": "user",
            "content": [
                *[{"type": "image", "image": img.convert("RGB")} for img in images],
                {"type": "text", "text": INSTRUCTION},
            ]
        }]

    # Official Qwen3-VL pattern (Qwen Team, 2025): apply_chat_template with
    # tokenize=True and return_dict=True processes images internally and
    # returns a ready-to-use tensor dict in a single call.
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = inputs.to(device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Trim input tokens from generated output (Qwen Team, 2025 pattern)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()


### 2.2. Evaluation Pipeline: Running Inference and Computing Metrics

This section defines a reusable evaluation pipeline that runs inference on all 70 test examples and computes all four metrics. Following the official Qwen3-VL inference pattern (Qwen Team, 2025), `processor.apply_chat_template` with `tokenize=True, return_dict=True` handles image processing and tokenization in a single call. The pipeline accepts the model, processor, and device as arguments, making it compatible with any prompt strategy (zero-shot, one-shot, or fine-tuned). A progress bar is printed every 10 examples for monitoring.


In [ ]:
def run_evaluation(model, processor, test_data, device="cpu",
                   max_new_tokens=256, verbose=True,
                   example_image=None, example_text=None):
    predictions = []
    references = [ex["text"] for ex in test_data]
    n = len(test_data)
    start_time = time.time()

    for i, ex in enumerate(test_data):
        if example_image is not None:
            images = [example_image.convert("RGB"), ex["image"].convert("RGB")]
        else:
            images = [ex["image"]]

        pred = predict(images, None, model, processor, device, max_new_tokens,
                      example_text=example_text)
        predictions.append(pred)

        if verbose and (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            print(f"  Processed {i+1}/{n} ({elapsed:.1f}s)")

    elapsed = time.time() - start_time
    if verbose:
        print(f"  Done. Total time: {elapsed:.1f}s ({elapsed/n:.1f}s per example)")

    return predictions, references, elapsed

## 3. Supervised Fine-Tuning

### 3.1. Qwen3-VL-2B-Instruct using LoRA

This section performs SFT on Qwen3-VL-2B-Instruct using LoRA (Hu et al., 2022). The goal is to adapt the model to recognize handwritten mathematical formulas and output them in LaTeX format. The Qwen-VL architecture supports dynamic-resolution processing (Wang et al., 2024), which may provide an advantage for the variable-height formula images in our datasets. A custom data collator applies the chat template and constructs labels using `process_vision_info` for Qwen3-VL's image handling, while LoRA keeps the training memory-efficient. After training, we evaluate all epoch checkpoints on the validation set (68 examples) and select the best one for final evaluation on the test set (70 examples).

This notebook trains on the combined dataset (1,864 examples, 2 epochs). Hyperparameters (batch size, gradient accumulation, learning rate, LoRA config) are identical to the single-dataset experiment in `3_qwen3vl_1dataset.ipynb`; only the dataset composition and number of epochs differ.

In [ ]:
# ── Experiment selector ───────────────────────────
# Change this flag to switch between training configurations.
#   False → LaTeX_OCR only (1,164 examples)
#   True  → LaTeX_OCR + MathWriting-human
USE_COMBINED = True
# ──────────────────────────────────

In [ ]:
# Training
NUM_EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4

In [ ]:
if USE_COMBINED:
    DATASET_NAME = "sft_2datasets"
    OUTPUT_DIR = "./qwen3vl_lora_combined"
    ADAPTER_SAVE_NAME = "qwen3vl_lora_2datasets"
    RESULTS_FILE = "qwen3vl_lora_2datasets_results.json"

    train_dataset = data_prep.get_combined_train(dataset, mw_train_subsample)
    print(f"Training on COMBINED dataset: {len(train_dataset):,} examples")
else:
    DATASET_NAME = "sft_1dataset"
    OUTPUT_DIR = "./qwen3vl_lora"
    ADAPTER_SAVE_NAME = "qwen3vl_lora_1dataset"
    RESULTS_FILE = "qwen3vl_lora_1dataset_results.json"

    train_dataset = dataset["train"]
    print(f"Training on LaTeX_OCR only: {len(train_dataset):,} examples")

train_dataset

Combined training set: 1,864 examples
Training on COMBINED dataset: 1,864 examples


Dataset({
    features: ['image', 'text'],
    num_rows: 1864
})

#### 3.1.1. LoRA Configuration and Model Loading

We use LoRA (Low-Rank Adaptation) to fine-tune only a small number of additional parameters while keeping the base model frozen. This drastically reduces VRAM usage and training time. The configuration targets all linear projection layers in the language model (q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj) with rank r=8. Rank r=8 is a commonly used value recommended by Hu et al. (2022) and has been shown to achieve a good balance between adaptation capacity and overfitting risk for parameter-efficient fine-tuning. Targeting all linear projections is the standard approach for adapting large vision-language models (Hu et al., 2022; Mangrulkar et al., 2022). The Instruct variant is used as the starting checkpoint because it already understands chat-based instructions.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_NAME)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
# LoRA config targeting all linear projection layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)

In [ ]:
# Load a fresh model and wrap it with LoRA adapters
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False  # Required when using gradient checkpointing with Trainer
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 8,716,288 || all params: 2,136,248,320 || trainable%: 0.4080


#### 3.1.2. Data Collation

A custom collator function is responsible for formatting each batch. For every training example, it constructs a two-turn conversation (user instruction with the image → assistant response with the ground-truth LaTeX) and applies the model's chat template. Unlike SmolVLM, Qwen3-VL requires images to be passed as actual PIL images in the message content (not placeholder dicts), and `process_vision_info` extracts the processed image/video tensors. Messages are constructed once and reused for both the chat template and `process_vision_info` to avoid duplication. Labels are derived from input_ids by masking padding tokens and `<|image_pad|>` tokens with -100 (which excludes them from the loss). This ensures the model learns to predict only the text tokens — both the instruction and the LaTeX response.


In [ ]:
# Image pad token for Qwen3-VL — mask it in labels so the model
# learns to predict only text tokens, not visual placeholders
image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")

def collate_fn(examples):
    """Formats a batch of (image, text) pairs into model inputs with labels."""
    all_messages = []
    for example in examples:
        image = example["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")

        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": (
                    "Convert this handwritten mathematical formula "
                    "to LaTeX format. Output only the LaTeX code, nothing else."
                )},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": example["text"]}]}
        ]
        all_messages.append(messages)

    # Build text via chat template (single pass, no duplicate construction)
    texts = [
        processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).strip()
        for msgs in all_messages
    ]

    # Extract processed image/video tensors via qwen_vl_utils
    batch_images = []
    batch_videos = []
    for msgs in all_messages:
        img_in, vid_in = process_vision_info(msgs)
        batch_images.extend(img_in)
        if vid_in:
            batch_videos.extend(vid_in)

    batch = processor(
        text=texts,
        images=batch_images,
        videos=batch_videos or None,
        return_tensors="pt",
        padding=True,
    )

    # Mask padding and image_pad tokens in labels (exclude from loss)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    batch["labels"] = labels
    return batch

#### 3.1.3. Training

Training hyperparameters (epochs, batch size, learning rate, etc.) are defined in the configuration cell at the top of this section. A checkpoint is saved after each epoch (`save_strategy="epoch"`) so that we can later select the best one on the validation set, effectively implementing a post-hoc early stopping strategy (Prechelt, 1998). The learning rate of 1e-4 is the default recommended for LoRA fine-tuning in the original LoRA paper (Hu et al., 2022) and has been adopted by most VLM fine-tuning recipes, including the SmolVLM fine-tuning guide. Training uses 2 epochs. The reduction from 3 (used in the single-dataset experiment) is motivated by the observation that training loss plateaued early on the single-dataset run, suggesting that 2 epochs suffice for the combined dataset as well. All other hyperparameters are kept identical for fair comparison. `remove_unused_columns=False` is required because the dataset contains image and text columns that are not direct model inputs but are needed by the custom collator. Effective batch size 16 is a practical choice dictated by GPU memory constraints (T4 GPU with 16 GB VRAM) and aligns with common practice in VLM fine-tuning literature.

In [ ]:
total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * 0.1))

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=3,
    bf16=True,
    output_dir=OUTPUT_DIR,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
)

trainer.train()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
25,3.565981
50,0.821283
75,0.154459
100,0.088593
125,0.075817
150,0.056051
175,0.070718
200,0.066904
225,0.054964


TrainOutput(global_step=234, training_loss=0.5311997447513107, metrics={'train_runtime': 15088.8292, 'train_samples_per_second': 0.247, 'train_steps_per_second': 0.016, 'total_flos': 6348569899057152.0, 'train_loss': 0.5311997447513107, 'epoch': 2.0})

#### 3.1.4. Checkpoint Selection on Validation Set (Prechelt, 1998)

Both epoch checkpoints are evaluated on the 68-example validation set. The best one is selected by normalized EM (primary) and CER (tiebreaker). The best checkpoint is then used for the final test evaluation.

In [ ]:
# Free the trained model to release GPU memory before evaluation
del model
gc.collect()
torch.cuda.empty_cache()

# Find all saved checkpoints
checkpoint_dirs = sorted(glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"Found {len(checkpoint_dirs)} checkpoints")

# Evaluate each checkpoint on the validation set
val_results = {}

for ckpt_dir in checkpoint_dirs:
    print(f"\nEvaluating {ckpt_dir} ...")

    # Load base model + LoRA adapter from this checkpoint
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        dtype="auto",
        device_map="auto",
    )

    model_ckpt = PeftModel.from_pretrained(base_model, ckpt_dir)
    model_ckpt.eval()

    preds, refs, _ = run_evaluation(
        model_ckpt, processor, dataset["validation"],
        device=device, verbose=False,
    )

    metrics = compute_metrics(preds, refs)
    epoch = ckpt_dir.split("-")[-1]
    val_results[epoch] = {
        "checkpoint": ckpt_dir,
        **metrics,
    }
    print(f"  EM: {metrics['exact_match']:.1%}, "
          f"EM (norm.): {metrics['exact_match_normalized']:.1%}, "
          f"CER: {metrics['cer']:.1%}, "
          f"BLEU: {metrics['bleu']:.1f}")

    # Free memory for next checkpoint
    del base_model, model_ckpt
    gc.collect()
    torch.cuda.empty_cache()

# Select best checkpoint by normalized EM, then CER as tiebreaker
best_epoch = max(val_results, key=lambda e: (val_results[e]["exact_match_normalized"],
                                              -val_results[e]["cer"]))
best_ckpt = val_results[best_epoch]["checkpoint"]
print(f"\nBest checkpoint: epoch {best_epoch} ({best_ckpt})")
print(f"  EM: {val_results[best_epoch]['exact_match']:.1%}, "
      f"EM (norm.): {val_results[best_epoch]['exact_match_normalized']:.1%}, "
      f"CER: {val_results[best_epoch]['cer']:.1%}, "
      f"BLEU: {val_results[best_epoch]['bleu']:.1f}")

Found 2 checkpoints

Evaluating ./qwen3vl_lora_combined/checkpoint-117 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  EM: 79.4%, EM (norm.): 79.4%, CER: 6.3%, BLEU: 92.4

Evaluating ./qwen3vl_lora_combined/checkpoint-234 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  EM: 83.8%, EM (norm.): 83.8%, CER: 4.1%, BLEU: 94.0

Best checkpoint: epoch 234 (./qwen3vl_lora_combined/checkpoint-234)
  EM: 83.8%, EM (norm.): 83.8%, CER: 4.1%, BLEU: 94.0


#### 3.1.5. Evaluation on Test Set

The selected checkpoint is evaluated on the same 70 test examples from linxy/LaTeX_OCR using the same metrics as in `3_qwen3vl_1dataset.ipynb`.

In [ ]:
print(f"Loading best checkpoint: {best_ckpt} ...")

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
)

model_sft = PeftModel.from_pretrained(base_model, best_ckpt)
model_sft.eval()

print("Running SFT evaluation on test set (70 examples)...")
sft_preds, sft_refs, sft_time = run_evaluation(
    model_sft, processor, dataset["test"],
    device=device,
)

Loading best checkpoint: ./qwen3vl_lora_combined/checkpoint-234 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Running SFT evaluation on test set (70 examples)...
  Processed 10/70 (60.9s)
  Processed 20/70 (116.1s)
  Processed 30/70 (175.7s)
  Processed 40/70 (233.4s)
  Processed 50/70 (285.6s)
  Processed 60/70 (345.9s)
  Processed 70/70 (402.8s)
  Done. Total time: 402.8s (5.8s per example)


In [ ]:
sft_metrics = compute_metrics(sft_preds, sft_refs)

print(f"\n{DATASET_NAME} results:")
print(f"  Exact Match:          {sft_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {sft_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {sft_metrics['cer']:.1%}")
print(f"  BLEU:                 {sft_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(5):
    print(f"  REF: {sft_refs[i]}")
    print(f"  PRD: {sft_preds[i]}")
    print()


sft_2datasets results:
  Exact Match:          85.7%
  Exact Match (norm.):  85.7%
  CER:                  3.3%
  BLEU:                 95.1

Examples:
  REF: \sqrt { b ^ { 2 } - 4 a c }
  PRD: \sqrt { b ^ { 2 } - 4 a c }

  REF: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
  PRD: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }

  REF: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
  PRD: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }

  REF: \lim _ { x \rightarrow - 1 } \frac { x ^ { 3 } + 1 } { x + 1 }
  PRD: \lim _ { x \rightarrow - 1 } \frac { x ^ { 3 } + 1 } { x + 1 }

  REF: 1 + \frac { 1 } { 1 ! } + \frac { 1 } { 2 ! } + \frac { 1 } { 3 ! } + \frac { 1 } { 4 ! }
  PRD: 1 + \frac { 1 } { 1 ! } + \frac { 1 } { 2 ! } + \frac { 1 } { 3 ! } + \frac { 1 } { 4 ! }



#### 3.1.6. Exporting Results

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

# Save LoRA adapter weights to Google Drive
adapter_save_path = os.path.join(SAVE_DIR, ADAPTER_SAVE_NAME)
model_sft.save_pretrained(adapter_save_path)
processor.save_pretrained(adapter_save_path)
print(f"LoRA adapter saved to: {adapter_save_path}")

# Save numerical results
results_this_step = {
    "predictions": sft_preds,
    "exact_match": sft_metrics["exact_match"],
    "exact_match_normalized": sft_metrics["exact_match_normalized"],
    "cer": sft_metrics["cer"],
    "bleu": sft_metrics["bleu"],
    "best_epoch": int(best_epoch),
    "validation_results": val_results,
    "inference_time_per_example": round(sft_time / len(dataset["test"]), 1),
}

save_path = os.path.join(SAVE_DIR, RESULTS_FILE)
with open(save_path, "w") as f:
    json.dump({"experiment": DATASET_NAME, "results": results_this_step}, f, ensure_ascii=False, indent=2)
print(f"Results saved to: {save_path}")

LoRA adapter saved to: /content/drive/MyDrive/latex_ocr/qwen3vl_lora_2datasets
Results saved to: /content/drive/MyDrive/latex_ocr/qwen3vl_lora_2datasets_results.json


## 4. Next Steps

All Qwen3-VL results (zero-shot, one-shot, SFT on LaTeX_OCR, and this combined-dataset SFT) are aggregated and analysed in Section 4 of `3_qwen3vl_1dataset.ipynb`.

## References

1. Hu, E.J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., Chen, W. (2022). LoRA: Low-Rank Adaptation of Large Language Models. *ICLR 2022*. https://arxiv.org/abs/2106.09685
2. Wang, P., Bai, S., Tan, S., et al. (2024). Qwen2-VL: Enhancing Vision-Language Model's Perception of the World at Any Resolution. *arXiv preprint*. https://arxiv.org/abs/2409.12191
3. Qwen Team. (2025). Qwen3-VL Technical Report. Alibaba Cloud. https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct
4. Brown, T.B., Mann, B., Ryder, N., et al. (2020). Language Models are Few-Shot Learners. *NeurIPS 2020*, 33, 1877–1901. https://arxiv.org/abs/2005.14165
5. Prechelt, L. (1998). Early Stopping — But When? In *Neural Networks: Tricks of the Trade*. Springer, LNCS 1524, pp. 55–69. https://doi.org/10.1007/3-540-49430-8_3
6. Mangrulkar, S., Gugger, S., Debut, L., Belkada, Y., Paul, S. (2022). PEFT: State-of-the-art Parameter-Efficient Fine-Tuning Methods. GitHub / HuggingFace. https://github.com/huggingface/peft
7. linxy. (2024). linxy/LaTeX_OCR (Human Handwrite subset). HuggingFace Datasets. https://huggingface.co/datasets/linxy/LaTeX_OCR
8. deepcopy. (2024). deepcopy/MathWriting-human. HuggingFace Datasets. https://huggingface.co/datasets/deepcopy/MathWriting-human